# Milestone 1 — Performance Benchmarking

This chapter presents the stored measurements across the runs listed in `book/data/benchmark.json`. Run environment strings identify Colab, but the retained summaries do not independently establish accelerator settings for every run. CPU-only execution is the required course testbed; missing evidence is reported explicitly below.

Tables and charts are generated during compilation from the stored measurement files. Local book compilation is not a new Colab crawling benchmark.


In [ ]:
import json
from pathlib import Path
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
from IPython.display import display, Markdown

# Set Plotly renderer for Jupyter Book
import plotly.io as pio
pio.renderers.default = "notebook_connected"

data_dir = Path("../../data")

# 1. Benchmark JSON
with open(data_dir / "benchmark.json", "r", encoding="utf-8") as f:
    bench = json.load(f)

# 2. Summary JSON
with open(data_dir / "summary.json", "r", encoding="utf-8") as f:
    summary = json.load(f)

# 3. Resource samples timeline
samples_path = data_dir / "resource_samples.csv"
df_samples = pd.read_csv(samples_path) if samples_path.exists() else pd.DataFrame()

# 4. Detailed papers (for per-paper elapsed time analysis)
papers_path = data_dir / "papers_detail.csv"
if papers_path.exists():
    df_detail = pd.read_csv(
        papers_path,
        dtype={"arxiv_id": "string", "member": "string", "account": "string", "run_id": "string"}
    )
else:
    df_detail = pd.DataFrame()

# Version-level outcomes support population-specific handoff counts.
df_versions = pd.read_csv(data_dir / "versions.csv", dtype={"arxiv_id": "string", "version": "string", "status": "string"})


## 1. Multi-Run Benchmark Results

Runtime is cumulative machine time across runs, not the elapsed time for the group to finish. Per-run RSS is measured by process-tree sampling; tracked disk may include corpus, staging, index and cache according to each run's configuration. Final notebook export/backup is outside the crawl benchmark; background checkpoint uploads are included. The table distinguishes retained-source byte counters from the sum of run disk measurements.


In [ ]:
multi_summary = bench.get('multi_run_summary', {})
total_runs = multi_summary.get('total_runs', len(bench.get('runs', [])))
total_wall = multi_summary.get('total_wall_seconds', bench.get('wall_seconds', 0.0))
median_wall = multi_summary.get('median_wall_seconds', bench.get('median_wall_seconds', 0.0))
total_discovery = multi_summary.get('total_discovery_elapsed_seconds', 0.0)
peak_ram = bench['ram_peak_bytes'] / (1024**2)
mean_ram = bench.get('ram_mean_bytes', 0) / (1024**2)
peak_disk = bench.get('disk_peak_bytes', 0) / (1024**2)
sum_run_disks = bench.get('sum_run_final_disk_bytes', bench.get('final_disk_bytes', 0)) / (1024**2)
merged_corpus_disk = bench.get('merged_corpus_tex_bib_bytes', 5424427008) / (1024**2)

avg_paper_time = summary.get('average_paper_processing_seconds', 0.0)
allocated_wall_per_id = total_wall / summary['papers_discovered']
paper_time_denom = summary.get('paper_time_denominator', 20000)

tot_hours = total_wall / 3600
med_mins = median_wall / 60
disc_hours = total_discovery / 3600

perf_summary = [
    {'Dimension': 'Multi-Run Scale', 'Metric': 'Total Pipeline Runs Documented', 'Observed Value': f'{total_runs:,} runs', 'Measurement Scope / Audit Note': '4 members * 3 accounts across Google Colab runtimes'},
    {'Dimension': 'Execution Time', 'Metric': 'Cumulative Machine-Hours (Wall Clock)', 'Observed Value': f'{tot_hours:,.1f} machine-hours ({total_wall:,.0f} s)', 'Measurement Scope / Audit Note': 'Summed wall time across all parallel & batch sessions'},
    {'Dimension': 'Execution Time', 'Metric': 'Total Entry Discovery Time (OAI-PMH)', 'Observed Value': f'{disc_hours:,.1f} machine-hours ({total_discovery:,.0f} s)', 'Measurement Scope / Audit Note': 'Cumulative wall time dedicated to batch metadata discovery'},
    {'Dimension': 'Execution Time', 'Metric': 'Median Run Wall Time', 'Observed Value': f'{med_mins:.1f} minutes ({median_wall:,.1f} s)', 'Measurement Scope / Audit Note': 'Typical duration of an assigned crawl session'},
    {'Dimension': 'Execution Time', 'Metric': 'Average Post-Discovery Time / Paper', 'Observed Value': f'{avg_paper_time:.2f} s / paper', 'Measurement Scope / Audit Note': f'Mean source download & reference processing latency (N = {paper_time_denom:,})'},
    {'Dimension': 'Execution Time', 'Metric': 'Allocated Run Wall Time per Unique ID', 'Observed Value': f'{allocated_wall_per_id:.2f} s / ID', 'Measurement Scope / Audit Note': 'Sum of recorded run wall times / unique IDs; includes discovery, repeated work and included checkpoint overhead'},
    {'Dimension': 'Memory Footprint', 'Metric': 'Max Observed Peak RAM (RSS)', 'Observed Value': f'{peak_ram:.1f} MiB', 'Measurement Scope / Audit Note': 'Maximum of stored per-run process-tree RSS peaks; brief spikes may be missed'},
    {'Dimension': 'Memory Footprint', 'Metric': 'Mean Run RAM Footprint', 'Observed Value': f'{mean_ram:.1f} MiB', 'Measurement Scope / Audit Note': 'Arithmetic mean of per-run sampled means; not weighted by run duration'},
    {'Dimension': 'Storage Footprint', 'Metric': 'Max Observed Tracked Disk Footprint', 'Observed Value': f'{peak_disk:.1f} MiB', 'Measurement Scope / Audit Note': 'Maximum tracked corpus/staging/index/cache bytes in a run; not just temporary files'},
    {'Dimension': 'Storage Footprint', 'Metric': 'Retained .tex/.bib Byte Counters', 'Observed Value': f'{merged_corpus_disk:,.1f} MiB ({merged_corpus_disk/1024:.2f} GiB)', 'Measurement Scope / Audit Note': 'Sum of retained-source byte counters for unique paper/version outcomes; excludes metadata and ZIP overhead'},
    {'Dimension': 'Storage Footprint', 'Metric': 'Sum of Run Final Disks (with overlap)', 'Observed Value': f'{sum_run_disks:,.1f} MiB ({sum_run_disks/1024:.2f} GiB)', 'Measurement Scope / Audit Note': 'Aggregate disk across 107 independent run staging folders'},
    {'Dimension': 'Storage Footprint', 'Metric': 'Complete Merged Output on Filesystem', 'Observed Value': 'Not measured', 'Measurement Scope / Audit Note': 'Requires one merged corpus measurement including metadata, references and receipts'},
    {'Dimension': 'Runtime Environment', 'Metric': 'CPU-only Evidence', 'Observed Value': 'All run flags confirm CPU-only' if bench.get('cpu_only') is True else 'At least one run reports an accelerator' if bench.get('cpu_only') is False else 'Not established by retained metadata', 'Measurement Scope / Audit Note': 'No default CPU-only assumption when flags are absent'},
    {'Dimension': 'Runtime Environment', 'Metric': 'Hardware / OS Specification', 'Observed Value': bench.get('environment') or 'Not recorded', 'Measurement Scope / Audit Note': 'Stored environment strings; accelerator flags are evaluated separately'}
]

df_perf = pd.DataFrame(perf_summary)
display(df_perf.style.hide(axis='index').set_properties(**{'text-align': 'left'}).set_table_attributes('class="dataframe table tex2jax_process"').set_table_styles([
    {'selector': 'th', 'props': [('background-color', '#1e3d59'), ('color', 'white'), ('font-weight', 'bold'), ('padding', '8px')]},
    {'selector': 'td', 'props': [('padding', '7px 10px')]},
    {'selector': 'tr:nth-child(even)', 'props': [('background-color', 'rgba(128, 128, 128, 0.05)')]}
]))


In [ ]:
run_metadata = json.loads((data_dir / 'run_metadata.json').read_text(encoding='utf-8'))
verified_runs = sum(bool(run.get('verified_archives')) and all(archive.get('verified') is True for archive in run['verified_archives']) for run in run_metadata)
run_flags = [run.get('cpu_only') for run in bench['runs']]
evidence = [
    {'Evidence': 'Runs with retained report-archive verification records', 'Coverage': f'{verified_runs} / {len(run_metadata)}'},
    {'Evidence': 'Runs explicitly marked CPU-only', 'Coverage': f"{sum(flag is True for flag in run_flags)} / {len(run_flags)}"},
    {'Evidence': 'Runs with recorded cache mode', 'Coverage': f"{sum(bool(run.get('cache_mode')) for run in bench['runs'])} / {len(run_flags)}"},
]
display(pd.DataFrame(evidence).style.hide(axis='index'))


## 2. Per-Paper Processing Time Distribution

To provide transparent performance granularity, the pipeline measures wall-clock seconds across distinct operational phases:
1. **Entry Discovery Phase**: Conducted via batch OAI-PMH queries prior to per-paper downloads (reported separately above as **Total Entry Discovery Time**).
2. **Per-Paper Ingestion Clock**: Starts when individual paper extraction begins (`paper_started`), capturing version tarball streaming, LaTeX whitelist filtering, and Semantic Scholar citation lookup.
3. **Checkpoint & Final Export**: Incremental checkpoint backups executed at 40-paper intervals.

The distribution below evaluates the **post-discovery per-paper ingestion duration** across all $N = 20,000$ assigned publications:


In [ ]:
if not df_detail.empty and 'elapsed_seconds' in df_detail.columns:
    p_times = df_detail['elapsed_seconds'].dropna()
    p_times_valid = p_times[p_times > 0]
    
    # 1. Histogram of Processing Times
    fig_time_hist = px.histogram(
        p_times_valid,
        nbins=60,
        labels={'value': 'Elapsed Time (Seconds)'},
        title='<b>Per-Paper Processing Time Distribution (N = 20,000 Papers)</b>',
        color_discrete_sequence=['#e67e22'],
        template='plotly_white'
    )
    fig_time_hist.update_layout(
        yaxis_title='Number of Papers',
        xaxis_title='Processing Time per Paper (Seconds)',
        height=420
    )
    fig_time_hist.show()
    
    # 2. Percentile Distribution Table (Context derived dynamically from data)
    p_percentiles = [
        {'Statistic / Percentile': 'Minimum Observed', 'Elapsed Seconds': f'{p_times_valid.min():.2f} s'},
        {'Statistic / Percentile': 'P25 (First Quartile)', 'Elapsed Seconds': f'{p_times_valid.quantile(0.25):.2f} s'},
        {'Statistic / Percentile': 'P50 (Median)', 'Elapsed Seconds': f'{p_times_valid.median():.2f} s'},
        {'Statistic / Percentile': 'P75 (Third Quartile)', 'Elapsed Seconds': f'{p_times_valid.quantile(0.75):.2f} s'},
        {'Statistic / Percentile': 'P90 (90th Percentile)', 'Elapsed Seconds': f'{p_times_valid.quantile(0.90):.2f} s'},
        {'Statistic / Percentile': 'P99 (99th Percentile)', 'Elapsed Seconds': f'{p_times_valid.quantile(0.99):.2f} s'},
        {'Statistic / Percentile': 'Mean (Arithmetic Average)', 'Elapsed Seconds': f'{p_times_valid.mean():.2f} s'},
        {'Statistic / Percentile': 'Maximum Observed', 'Elapsed Seconds': f'{p_times_valid.max():.2f} s'}
    ]
    
    display(Markdown('#### Processing Time Statistical Percentiles'))
    display(pd.DataFrame(p_percentiles).style.hide(axis='index').set_properties(**{'text-align': 'left'}).set_table_attributes('class="dataframe table tex2jax_process"').set_table_styles([{'selector': 'th', 'props': [('background-color', '#2c3e50'), ('color', 'white'), ('font-weight', 'bold')]}, {'selector': 'tr:nth-child(even)', 'props': [('background-color', 'rgba(128, 128, 128, 0.05)')]}]))
    
    # 3. Top 5 slowest papers (Outlier inspection)
    top_slowest = df_detail.sort_values(by='elapsed_seconds', ascending=False).head(5)
    cols_to_show = [c for c in ['arxiv_id', 'member', 'versions_expected', 'total_references', 'kept_references', 'elapsed_seconds', 'title'] if c in top_slowest.columns]
    
    display(Markdown('#### Top 5 Publications with Highest Ingestion Latency'))
    # Normalize superscript math symbols for seamless MathJax typography
    top_slowest_display = top_slowest.copy()
    if 'title' in top_slowest_display.columns:
        top_slowest_display['title'] = top_slowest_display['title'].str.replace(r'\$\^\\?ast\$', '*', regex=True)

    display(top_slowest_display[cols_to_show].style.hide(axis='index')\
        .set_properties(**{'text-align': 'left'})\
        .set_table_attributes('class="dataframe table tex2jax_process"')\
        .set_table_styles([
        {'selector': 'th', 'props': [('background-color', '#c0392b'), ('color', 'white'), ('font-weight', 'bold')]},
        {'selector': 'tr:nth-child(even)', 'props': [('background-color', 'rgba(128, 128, 128, 0.05)')]}
    ]))

## 3. Resource Footprint Over Time

The plot shows the longest retained trace as one representative run, selected by its largest elapsed timestamp. The aggregation script thins traces for display, so these CSV points do not preserve every original 0.25-second sample or necessarily contain each run's recorded peak. Global peak statistics use the per-run benchmark summaries instead.


In [ ]:
if not df_samples.empty:
    # Distinguish runs and calculate cumulative timeline across sessions
    df_samples_clean = df_samples.copy()
    df_samples_clean['ram_mib'] = df_samples_clean['ram_bytes'] / (1024**2)
    df_samples_clean['disk_mib'] = df_samples_clean['disk_bytes'] / (1024**2)
    
    # Select the longest representative run for detailed trace analysis
    run_lengths = df_samples_clean.groupby('run_id')['elapsed_seconds'].max()
    longest_run_id = run_lengths.idxmax()
    df_rep = df_samples_clean[df_samples_clean['run_id'] == longest_run_id].sort_values('elapsed_seconds')
    df_rep['time_min'] = df_rep['elapsed_seconds'] / 60.0
    
    fig = make_subplots(specs=[[{'secondary_y': True}]])
    
    fig.add_trace(
        go.Scatter(
            x=df_rep['time_min'],
            y=df_rep['ram_mib'],
            name='RAM Usage (MiB)',
            line=dict(color='#e67e22', width=2.0)
        ),
        secondary_y=False,
    )
    
    fig.add_trace(
        go.Scatter(
            x=df_rep['time_min'],
            y=df_rep['disk_mib'],
            name='Disk Footprint (MiB)',
            line=dict(color='#2980b9', width=2.0, dash='dash')
        ),
        secondary_y=True,
    )
    
    fig.update_layout(
        title=f'<b>Resource Footprint Over Time — Representative Crawl Session ({longest_run_id[:8]}…)</b>',
        xaxis_title='Elapsed Session Time (Minutes)',
        template='plotly_white',
        height=480,
        hovermode='x unified',
        legend=dict(yanchor='top', y=0.99, xanchor='left', x=0.01)
    )
    
    fig.update_yaxes(title_text='<b>RAM Footprint (MiB)</b>', secondary_y=False)
    fig.update_yaxes(title_text='<b>Disk Footprint (MiB)</b>', secondary_y=True)
    fig.show()
    
    # Use the original run-summary peaks, not the thinned plotting trace.
    recorded_peaks = pd.to_numeric(pd.Series([run.get('ram_peak_bytes') for run in bench['runs']]), errors='coerce').dropna() / 1024**2
    trace_peak = df_samples_clean['ram_mib'].max()
    display(Markdown(f"> **RSS measurement scope:** The {len(recorded_peaks):,} run summaries report a maximum peak of **{peak_ram:.1f} MiB** and a median run peak of **{recorded_peaks.median():.1f} MiB**. The thinned plotting CSV reaches **{trace_peak:.1f} MiB**; it is not the source for the global peak. Sampling can miss brief allocations between observations."))

else:
    print('No timeline samples recorded.')


## 4. Analysis and Measurement Limits

The discussion below is generated from the same stored summary values as the benchmark table.


In [ ]:
cached_versions = int(df_versions['used_cache'].astype(str).str.lower().eq('true').sum())
size_before = summary['average_source_bytes_before'] / 1024
size_after = summary['average_source_bytes_after'] / 1024
size_n = summary['source_size_denominator']
reduction = summary['size_reduction_ratio'] * 100

display(Markdown(f"""### Running time
The documented runs total **{tot_hours:,.1f} machine-hours**, including **{disc_hours:,.1f} machine-hours** of entry discovery. This total includes repeated attempts and is not the group's calendar completion time. Mean post-discovery processing is **{avg_paper_time:.2f} seconds per selected paper record**. Allocated run wall time is **{allocated_wall_per_id:.2f} seconds per unique ID**, including discovery and overhead within the recorded run boundaries. It is an aggregate cost, not an individual paper latency.

The current crawler configuration waits 3.1 seconds before arXiv/S2 requests and allows up to three attempts, with server Retry-After taking precedence. Request pacing and network work can contribute to elapsed time, but the stored telemetry does not quantify their shares or CPU utilization.

### RAM and cache
The run summaries report a maximum observed RSS of **{peak_ram:.1f} MiB**. Sampling may miss short peaks; these observations do not guarantee memory bounds or absence of future OOM failures. The retained metadata does not establish the actual RAM/disk capacity of every runtime.

There are **{cached_versions:,} selected source versions marked used_cache=True**. Original per-run cache modes and accelerator flags are not preserved in all retained summaries; they must be recovered from the original archives to establish historical execution settings. A CPU-only Colab benchmark remains to be documented; cache use must be stated separately.

### Storage
Across the same **{size_n:,} resolved versions**, mean uncompressed source bytes change from **{size_before:,.1f} KiB** to **{size_after:,.1f} KiB**, a total byte-volume reduction of **{reduction:.2f}%**. Filtering excludes auxiliary files as well as figures.

The sum of retained .tex/.bib byte counters for unique versions is **{merged_corpus_disk/1024:.2f} GiB**. This excludes metadata, references, receipts and archive overhead. A complete merged-output filesystem measurement is not available. The sum of final disk measurements across runs is **{sum_run_disks/1024:.2f} GiB** and may include overlapping content and other work files. It is not a single merged filesystem measurement. The maximum tracked disk observation in one run is **{peak_disk/1024:.2f} GiB**; comparing it to Colab capacity requires that runtime's actual capacity metadata.

Only data with committed, verified backup receipts is protected by the checkpoint workflow. Pending or uploading groups may need to be collected again after a runtime is lost.
"""))


## 5. Downstream Corpus Readiness and Handoff

The following populations are calculated from `papers_detail.csv` and `versions.csv`. Source populations are disjoint; the reference populations are a separate view and may overlap the source groups. A resolved source outcome does not certify successful parsing, bibliography matching or labeled ground truth.


In [ ]:
counts = df_detail[['versions_expected', 'versions_success', 'versions_no_tex', 'versions_failed', 'versions_pending']].apply(pd.to_numeric, errors='raise')
full_tex = (counts['versions_expected'] > 0) & (counts['versions_success'] == counts['versions_expected'])
resolved = (counts['versions_expected'] > 0) & ((counts['versions_success'] + counts['versions_no_tex']) == counts['versions_expected'])
resolved_with_no_tex = resolved & (counts['versions_no_tex'] > 0)
all_no_tex = resolved & (counts['versions_no_tex'] == counts['versions_expected'])
mixed_resolved = resolved_with_no_tex & ~all_no_tex
unresolved = ~resolved
refs_total = pd.to_numeric(df_detail['total_references'], errors='raise')
refs_kept = pd.to_numeric(df_detail['kept_references'], errors='raise')
arxiv_pool = refs_kept > 0
non_arxiv_only = df_detail['reference_status'].eq('success') & refs_total.gt(0) & refs_kept.eq(0)

def population_versions(mask):
    return df_versions[df_versions['arxiv_id'].isin(df_detail.loc[mask, 'arxiv_id'])]

def paper_volume(mask):
    return f"{int(mask.sum()):,} papers ({mask.mean()*100:.2f}%)"

full_versions = population_versions(full_tex)
no_tex_versions = population_versions(resolved_with_no_tex)
incomplete_versions = population_versions(unresolved)
handoff_rows = [
    {'Population': 'Full-TeX publications', 'Paper scope': paper_volume(full_tex), 'Version / citation scope': f'{len(full_versions):,} versions in these papers', 'M2 handoff': 'Parse each version and locate the actual compilation root; a .bib file is not guaranteed.', 'M3 handoff': 'Parse bibliographic entries, then create labels and evaluate matching.'},
    {'Population': 'Resolved papers with at least one no_tex version', 'Paper scope': paper_volume(resolved_with_no_tex), 'Version / citation scope': f"{int(no_tex_versions['status'].eq('no_tex').sum()):,} no_tex versions in this group", 'M2 handoff': 'Retain empty versions; parse usable TeX versions of mixed papers.', 'M3 handoff': 'Reference-job outcomes must be checked independently of source outcomes.'},
    {'Population': 'Unresolved source papers', 'Paper scope': paper_volume(unresolved), 'Version / citation scope': f"{int(incomplete_versions['status'].eq('failed').sum()):,} failed and {int(incomplete_versions['status'].eq('pending').sum()):,} pending versions", 'M2 handoff': 'Keep the missing-version audit and review recoverable failures.', 'M3 handoff': 'Do not infer reference availability from a source-download failure.'},
    {'Population': 'Papers with a non-empty arXiv reference pool', 'Paper scope': paper_volume(arxiv_pool), 'Version / citation scope': f'{int(refs_kept[arxiv_pool].sum()):,} kept citation rows', 'M2 handoff': 'Candidate external metadata for parsed bibliography entries.', 'M3 handoff': 'Candidate pool only; manual/automatic labeling and publication-level splits are still required.'},
    {'Population': 'Successful reference jobs with only non-arXiv citations', 'Paper scope': paper_volume(non_arxiv_only), 'Version / citation scope': f'{int(refs_total[non_arxiv_only].sum()):,} citations in these papers', 'M2 handoff': 'Preserve bibliography entries and all-reference metadata.', 'M3 handoff': 'These citations have no arXiv-keyed candidate in this pool.'},
]
display(pd.DataFrame(handoff_rows).style.hide(axis='index').set_properties(**{'text-align':'left'}))
display(Markdown(f"Of the resolved source papers, **{int(all_no_tex.sum()):,}** have no usable TeX in any version and **{int(mixed_resolved.sum()):,}** contain both TeX and no_tex versions. The corpus-wide non-arXiv reference count is **{summary['non_arxiv_reference_rows']:,}**, including non-arXiv citations in mixed-reference papers; it is distinct from the non-arXiv-only row above."))
display(Markdown("### Transition to M2 and M3\n\nM2 must inspect source files and per-version JSON receipts at `sources/.vN.receipt.json`; successful source extraction does not establish a standard `main.tex` root or parser success. M3 must label BibTeX entries, split by publication and evaluate MRR@5 according to the course specification. No acyclicity, temporal ordering or graph-model suitability has been established by the M1 crawl statistics."))
